# AC-MOT v10_p4 — FP16 Fair Benchmark + Deployment Validation

This notebook does **not** modify v10_p3. It runs the new FP16 fair-comparison protocol and saves a separate v10_p4 result folder.

After the benchmark, use `drone_runtime_v10_p4.py` on recorded flight video, webcam, or an RTSP drone camera stream.

In [ ]:
# CELL 1 — GPU + INSTALL + MOUNT DRIVE
!nvidia-smi
!pip install -q ultralytics==8.3.200 motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml
!pip install -q git+https://github.com/JonathonLuiten/TrackEval.git
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
# CELL 2 — GET v10_p4 CODE
import os, subprocess, pathlib
REPO='/content/ACMOT-Codex-V10Style-Portable'
if pathlib.Path(REPO).exists():
    subprocess.run(['git','-C',REPO,'pull'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/AhmedCode110/ACMOT-Codex-V10Style-Portable.git',REPO],check=True)
print('Repo ready:', REPO)


In [ ]:
# CELL 3 — VERIFY + STAGE DATASET TO LOCAL /content
from pathlib import Path
import shutil, pandas as pd
from tqdm.auto import tqdm
DRIVE_DATASET=Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
LOCAL=Path('/content/visdrone_v10_p4_local/VisDrone2019-MOT-test-dev')
SEQ=DRIVE_DATASET/'sequences'; ANN=DRIVE_DATASET/'annotations'
seqs=sorted([p.name for p in SEQ.iterdir() if p.is_dir()])
assert len(seqs)==17, f'Expected 17 sequences, found {len(seqs)}'
for s in seqs:
    frames=sorted((SEQ/s).glob('*.jpg'))
    gt=pd.read_csv(ANN/f'{s}.txt',header=None)
    assert len(frames)==int(gt.iloc[:,0].max()), f'Mismatch {s}'
if LOCAL.exists(): shutil.rmtree(LOCAL)
(LOCAL/'sequences').mkdir(parents=True); (LOCAL/'annotations').mkdir(parents=True)
total=sum(len(list((SEQ/s).glob('*.jpg'))) for s in seqs)+17
p=tqdm(total=total,desc='Drive -> /content',dynamic_ncols=True)
for s in seqs:
    dst=LOCAL/'sequences'/s; dst.mkdir()
    for fp in sorted((SEQ/s).glob('*.jpg')):
        shutil.copy2(fp,dst/fp.name); p.update(1)
    shutil.copy2(ANN/f'{s}.txt',LOCAL/'annotations'/f'{s}.txt'); p.update(1)
p.close()
print('Local dataset ready:', LOCAL)


In [ ]:
# CELL 4 — RUN COMPLETE FP16 FAIR BENCHMARK + OFFICIAL TRACKEVAL
from datetime import datetime
from pathlib import Path
import subprocess, sys
OUT=Path('/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR') / ('codex_v10_p4_fp16_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
cmd=[sys.executable, f'{REPO}/fair_benchmark_v10_p4.py', '--dataset', str(LOCAL), '--weights', '/content/yolov8n.pt', '--output', str(OUT), '--run-trackeval']
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True)
print('FINAL RESULT FOLDER:', OUT)


In [ ]:
# CELL 5 — OPTIONAL DEPLOYMENT SIMULATION ON A RECORDED FLIGHT VIDEO
# Put a video path below and run this cell. This uses the exact same v10_p4 AC-MOT controller.
VIDEO='/content/drive/MyDrive/your_flight_video.mp4'
DEPLOY_OUT='/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR/drone_simulation'
# Uncomment after setting VIDEO:
# !python {REPO}/drone_runtime_v10_p4.py --source "{VIDEO}" --weights /content/yolov8n.pt --output-dir "{DEPLOY_OUT}" --save-video
